# DRAFT — 04c human immune de novo annotation

**Presentation scaffold only.** Generated 2026-08-10 from the committed decision
record `/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/resources/annotations/human_immune.csv` and locked vocabulary `/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/resources/annotation_vocabulary.yaml`. The map below is
frozen from the CSV; this notebook does not infer, edit, or write annotations.

- resolution: `leiden_scVI_3.0`
- decision CSV SHA-256: `8c24e61758fbc590e1bd8393fad26d028dbd6239edd024b065c57162474f2433`
- vocabulary SHA-256: `df2709ede865ad407aed85073f8957bab5c09c5be829c11ef4e10e0804566666`


## 1. Frozen configuration and annotation map

In [ ]:
SPECIES = 'human'
LINEAGE = 'immune'
RES = 'leiden_scVI_3.0'
REPO = '/n/lab_storage/eddy_lab/users/nhilgert/xmens_final'
CSV_PATH = '/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/resources/annotations/human_immune.csv'
VOCAB_PATH = '/n/lab_storage/eddy_lab/users/nhilgert/xmens_final/resources/annotation_vocabulary.yaml'
CSV_SHA256 = '8c24e61758fbc590e1bd8393fad26d028dbd6239edd024b065c57162474f2433'
VOCAB_SHA256 = 'df2709ede865ad407aed85073f8957bab5c09c5be829c11ef4e10e0804566666'
ANNOTATION = {
    "0": "macrophage",
    "1": "EXCLUDE",
    "10": "t_cell_cd8",
    "11": "macrophage",
    "12": "t_cell_cd8",
    "13": "mast_cell",
    "14": "t_cell_regulatory",
    "15": "t_cell_cd4",
    "16": "t_cell_regulatory",
    "17": "t_cell_cd4",
    "18": "t_cell_cd4",
    "19": "t_cell_cd4",
    "2": "macrophage",
    "20": "t_cell_cd4",
    "21": "unk_cell_gzmb_high_itgax_high",
    "22": "t_cell_cd8",
    "23": "t_cell_cd8",
    "24": "t_cell_cd8",
    "25": "t_cell_cd8",
    "26": "t_cell_cd8",
    "27": "t_cell_cd8",
    "3": "EXCLUDE",
    "4": "macrophage",
    "5": "t_cell_cd8",
    "6": "unk_cell_gzmb_high_itgax_high",
    "7": "t_cell_gamma_delta",
    "8": "unk_cell_gzmb_low_itgax_low",
    "9": "EXCLUDE"
}
MARKERS = {
    "pan": [
        "PTPRC"
    ],
    "myeloid": [
        "CD68",
        "CSF1R",
        "ITGAM",
        "LYZ",
        "ITGAX"
    ],
    "lymph": [
        "CD3E",
        "CD8A",
        "CD4",
        "FOXP3"
    ],
    "nk": [
        "NCAM1",
        "NCR1",
        "KLRD1",
        "GZMB"
    ],
    "mast": [
        "TPSAB1",
        "CPA3",
        "MS4A2",
        "KIT"
    ],
    "contam": [
        "CDH1",
        "PDGFRA",
        "PECAM1",
        "ACTA2"
    ]
}

## 2. Load the frozen lineage latent and verify complete cluster coverage

In [ ]:
import hashlib, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import pandas as pd, scanpy as sc

REPO = Path(REPO)
LATENT = REPO / "data/interim" / f"latent_{SPECIES}_{LINEAGE}.h5ad"
adata = sc.read_h5ad(LATENT)
assert RES in adata.obs, f"{RES} absent from {LATENT}"
clusters = set(adata.obs[RES].astype(str).unique())
assert clusters == set(ANNOTATION), (
    f"frozen map mismatch: missing={sorted(clusters-set(ANNOTATION))}, "
    f"extra={sorted(set(ANNOTATION)-clusters)}")
print(f"{adata.n_obs:,} cells x {adata.n_vars:,} genes; {len(clusters)} clusters")

## 3. Raw marker dotplot by frozen cluster

In [ ]:
plot_sets = {name: [gene for gene in genes if gene in adata.var_names]
             for name, genes in MARKERS.items()}
plot_sets = {name: genes for name, genes in plot_sets.items() if genes}
if plot_sets:
    sc.pl.dotplot(adata, plot_sets, groupby=RES, standard_scale="var",
                  figsize=(10, max(6, 0.22 * len(clusters))), show=True)
else:
    print("No configured marker genes are present.")

## 4. Apply the committed map in memory using the shared producer

In [ ]:
import sys
sys.path.insert(0, str(REPO / "workflow/scripts"))
from apply_annotation import (apply_map, build_vocabulary_rollup,
                              validate_canonical_annotations)
import yaml

vocab = yaml.safe_load(Path(VOCAB_PATH).read_text())
rollup = build_vocabulary_rollup(vocab)
summary = apply_map(adata, RES, ANNOTATION, rollup=rollup, output_mode="fine")
kept = adata[~adata.obs.excluded].copy()
validate_canonical_annotations(kept, vocab, context=f"{SPECIES}/{LINEAGE} draft")
kept.obs["display_label"] = kept.obs.cell_subtype.astype(str).where(
    kept.obs.cell_subtype.astype(str).ne(""), kept.obs.cell_type.astype(str))
print(summary.to_string())
print("\ncanonical cell types:")
print(kept.obs.cell_type.value_counts().to_string())
print("\ncanonical subtypes (nonempty):")
print(kept.obs.loc[kept.obs.cell_subtype.ne(""), "cell_subtype"].value_counts().to_string())

## 5. Verification dotplot ordered by the frozen annotation

In [ ]:
if plot_sets:
    sc.pl.dotplot(kept, plot_sets, groupby="display_label", standard_scale="var",
                  figsize=(10, max(6, 0.35 * kept.obs.display_label.nunique())), show=True)
sc.pl.embedding(kept, "X_umap_scVI", color=[RES, "display_label"],
                legend_loc="right margin", show=True)

## 6. Freeze check — assert the committed inputs are unchanged

In [ ]:
def _sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
assert _sha(CSV_PATH) == CSV_SHA256, "decision CSV changed; regenerate this draft"
assert _sha(VOCAB_PATH) == VOCAB_SHA256, "vocabulary changed; regenerate this draft"
now = pd.read_csv(CSV_PATH, dtype=str)
now_map = dict(zip(now.cluster.str.strip(), now.label.str.strip()))
assert now_map == ANNOTATION, "embedded map differs from committed CSV"
print("PASS: CSV and vocabulary hashes match; embedded map is exact.")